In [0]:
import re

dbutils.widgets.text("catalog", "appletvshow")
dbutils.widgets.text("category", "Drama")
CATALOG  = dbutils.widgets.get("catalog")
category = dbutils.widgets.get("category")

# Clean the name for use in a table name: "Docu-series & Specials" -> "docu_series_specials"
suffix = re.sub(r"[^a-z0-9]+", "_", category.lower()).strip("_")
table  = f"{CATALOG}.gold.cat_{suffix}"

spark.sql(
    f"CREATE OR REPLACE TABLE {table} AS "
    f"SELECT * FROM {CATALOG}.silver.wiki_shows WHERE category = :category",
    args={"category": category},
)

n = spark.table(table).count()
print(f"{table}: {n} rows")
if n == 0:
    raise ValueError(f"No rows for category '{category}' — check spelling")